In [1]:
import joblib
import pandas as pd
import numpy as np
import sys

# utils, submit에서 필요한 함수 로드
import importlib
import utils as ut
importlib.reload(ut)

import submit as sb
importlib.reload(sb)

import save_log as sv
importlib.reload(sv)


import elo
importlib.reload(elo)

<module 'elo' from 'c:\\Users\\User\\Desktop\\statiz_prediction\\elo.py'>

In [2]:
# 모델 로드
loaded_model = joblib.load("logit_model.pkl")
feature_cols = joblib.load("feature_cols.pkl")

loaded_model_sp = joblib.load("logit_model_sp.pkl")
feature_cols_sp = joblib.load("feature_cols_sp.pkl")
print("모델 로드 완료")

모델 로드 완료


In [11]:
loaded_model_core = joblib.load("logit_model_reduced_core.pkl")
feature_cols_core = joblib.load("feature_cols_reduced_core.pkl")
loaded_model_core_sp = joblib.load("logit_model_reduce_core_sp.pkl")
feature_cols_core_sp = joblib.load("feature_cols_recent_reduce_core_sp.pkl")
print("코어 모델 로드 완료")

코어 모델 로드 완료


In [3]:
loaded_model_elo = joblib.load("logit_model_elo_full.pkl")
feature_cols_elo = joblib.load("feature_cols_elo_full.pkl")
loaded_model_elo_sp = joblib.load("logit_model_elo_sp.pkl")
feature_cols_elo_sp = joblib.load("feature_cols_elo_sp.pkl")
print("ELO 모델 로드 완료")


ELO 모델 로드 완료


In [17]:
print(feature_cols_core_sp)
print(feature_cols_elo_sp)

['sp_FIP_diff', 'sp_ERA_diff', 'sp_K9_diff', 'sp_BB9_diff', 'sp_HR9_diff', 'sp_KBB_diff', 'recent_10_win_rate_diff', 'recent_5_run_diff_avg_diff', 'recent_5_runs_against_avg_diff', 'rest_days_diff', 'prev_game_win_diff']
['sp_ERA_diff', 'sp_FIP_diff', 'sp_WHIP_diff', 'sp_K9_diff', 'sp_BB9_diff', 'sp_HR9_diff', 'sp_KBB_diff', 'recent_5_result_avg_diff', 'recent_5_runs_for_avg_diff', 'recent_5_runs_against_avg_diff', 'recent_5_run_diff_avg_diff', 'recent_10_result_avg_diff', 'recent_10_runs_for_avg_diff', 'recent_10_runs_against_avg_diff', 'recent_10_run_diff_avg_diff', 'rest_days_diff', 'prev_game_result_diff', 'home_elo_pre', 'away_elo_pre', 'elo_diff', 'elo_exp_home']


### API에서 오늘 경기 데이터 불러오기

### Given lineup
- 날짜 변경 필수

In [4]:
# 예측 데이터 구축
today = pd.Timestamp.now(tz="Asia/Seoul").normalize().tz_localize(None)
pred_result = ut.build_prediction_data_for_date(  
    date_str=today.strftime("%Y-%m-%d"),  # 직접 지정 시 예: "2026-06-01"
    # date_str="2026-06-01",
    sleep_sec=0.12,
    verbose=True
)

pred_df = pred_result["pred_df"]


=== 1. collect_games ===
[games] 2026-06-02: 5 rows
games_df.shape = (5, 21)


,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_score,home_win,game_state,game_time,weather,temperature,humidity,windDirection,windSpeed,rainprobability
0,20260281,2026-06-02 09:30:00,2026,6002,7002,1001,15143,16572,벤자민,박준영,...,None,0,1,18:30:00,0,0,0,0,0,0
1,20260282,2026-06-02 09:30:00,2026,9002,10001,2001,16446,13941,베니지아노,알칸타라,...,None,0,1,18:30:00,0,0,0,0,0,0
2,20260283,2026-06-02 09:30:00,2026,12001,5002,2002,14776,10652,한차현,임찬규,...,None,0,1,18:30:00,0,0,0,0,0,0
3,20260284,2026-06-02 09:30:00,2026,1001,11001,7003,15531,16454,후라도,토다,...,None,0,1,18:30:00,0,0,0,0,0,0
4,20260285,2026-06-02 09:30:00,2026,2002,3001,6001,16088,12930,네일,나균안,...,None,0,1,18:30:00,0,0,0,0,0,0


=== 2. filter target games ===
games_pred.shape = (5, 21)


,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_score,home_win,game_state,game_time,weather,temperature,humidity,windDirection,windSpeed,rainprobability
0,20260281,2026-06-02 09:30:00,2026,6002,7002,1001,15143,16572,벤자민,박준영,...,None,0,1,18:30:00,0,0,0,0,0,0
1,20260282,2026-06-02 09:30:00,2026,9002,10001,2001,16446,13941,베니지아노,알칸타라,...,None,0,1,18:30:00,0,0,0,0,0,0
2,20260283,2026-06-02 09:30:00,2026,12001,5002,2002,14776,10652,한차현,임찬규,...,None,0,1,18:30:00,0,0,0,0,0,0
3,20260284,2026-06-02 09:30:00,2026,1001,11001,7003,15531,16454,후라도,토다,...,None,0,1,18:30:00,0,0,0,0,0,0
4,20260285,2026-06-02 09:30:00,2026,2002,3001,6001,16088,12930,네일,나균안,...,None,0,1,18:30:00,0,0,0,0,0,0


=== 3. collect_lineups ===
[lineup] 20260281: 2 rows
[lineup] 20260282: 2 rows
[lineup] 20260283: 2 rows
[lineup] 20260284: 2 rows
[lineup] 20260285: 2 rows
lineup_df.shape = (10, 13)
=== 4. collect_rosters ===
[roster] 2026-06-02 team=1001: 29 rows
[roster] 2026-06-02 team=2002: 29 rows
[roster] 2026-06-02 team=3001: 29 rows
[roster] 2026-06-02 team=5002: 29 rows
[roster] 2026-06-02 team=6002: 29 rows
[roster] 2026-06-02 team=7002: 29 rows
[roster] 2026-06-02 team=9002: 29 rows
[roster] 2026-06-02 team=10001: 29 rows
[roster] 2026-06-02 team=11001: 29 rows
[roster] 2026-06-02 team=12001: 30 rows
roster_df.shape = (291, 5)
n_unique_player_ids = 291
=== 5. collect_player_season ===
[playerSeason] 10892: OK
[playerSeason] 10363: OK
[playerSeason] 10640: OK
[playerSeason] 10232: OK
[playerSeason] 14113: OK
[playerSeason] 14116: OK
[playerSeason] 14618: OK
[playerSeason] 12936: OK
[playerSeason] 15034: OK
[playerSeason] 12934: OK
[playerSeason] 10894: OK
[playerSeason] 11422: OK
[playerSea

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_sp_KBB,away_sp_QS,away_sp_GS,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff
0,20260281,2026-06-02 09:30:00,2026,6002,7002,1001,15143,16572,벤자민,박준영,...,2.0000,0.0,2.0,-0.94,-3.07908,0.01,-0.137,-1.158,-1.914,0.9091
1,20260282,2026-06-02 09:30:00,2026,9002,10001,2001,16446,13941,베니지아노,알칸타라,...,6.7778,6.0,10.0,2.45,1.76224,0.51,-1.842,2.639,-0.025,-5.0778
2,20260283,2026-06-02 09:30:00,2026,12001,5002,2002,14776,10652,한차현,임찬규,...,1.8125,3.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20260284,2026-06-02 09:30:00,2026,1001,11001,7003,15531,16454,후라도,토다,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20260285,2026-06-02 09:30:00,2026,2002,3001,6001,16088,12930,네일,나균안,...,2.6667,4.0,10.0,0.39,-0.99879,-0.23,-0.714,-1.405,-0.673,2.1333


In [5]:
pred_df.isna().sum().sum()  # 정상은 10, full 15

np.int64(42)

### 선발투수 결측치 보간


In [6]:
season_pit_df = pd.read_csv('seaon_pit_df.csv')

pred_df = ut.fill_and_recompute_sp_features(
    pred_df=pred_df,
    season_pit_df=season_pit_df,
    target_year=2026
)

In [6]:
pred_df

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_sp_GS,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,home_sp_imputed,away_sp_imputed
0,20260213,2026-05-16 05:00:00,2026,12001,7002,2002,11394,16422,배제성,화이트,...,1.0,-3.86,-2.01857,-1.24,5.786,-1.157,0.000,3.0000,0,0
1,20260211,2026-05-16 08:00:00,2026,6002,3001,1001,16299,11310,잭로그,박세웅,...,7.0,-1.04,-1.66501,-0.19,0.455,-2.538,-0.329,3.1111,0,0
2,20260212,2026-05-16 08:00:00,2026,9002,5002,2001,13126,14134,최민준,이정용,...,2.0,-1.35,-0.04270,-0.15,2.536,1.423,0.155,0.1008,0,0
3,20260214,2026-05-16 08:00:00,2026,1001,2002,7003,13574,14788,오러클린,이의리,...,8.0,-5.62,-3.14107,-0.91,-1.286,-4.532,-1.704,1.1221,0,0
4,20260215,2026-05-16 08:00:00,2026,11001,10001,8005,11353,16339,구창모,로젠버그,...,NaN,0.73,1.56240,0.02,-2.808,-0.427,0.800,-0.5636,0,1


In [7]:
pred_df.isna().sum().sum()  # Encoding-normalized comment.

np.int64(14)

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,home_sp_imputed,away_sp_imputed,home_win_proba
0,20260111,2026-04-23 09:30:00,2026,5002,7002,1001,14134,16107,이정용,황준서,...,1.76,2.69464,0.33,-7.232,-0.643,0.000,-1.4167,0,0,0.254640
1,20260112,2026-04-23 09:30:00,2026,10001,11001,1003,16127,11353,김연주,구창모,...,3.80,2.44236,0.75,-1.168,1.131,0.936,-0.7550,1,0,0.274754
2,20260113,2026-04-23 09:30:00,2026,12001,2002,2002,14600,14788,소형준,이의리,...,-2.74,-3.18603,-0.84,-1.537,-6.266,-1.158,4.1667,0,0,0.651515
3,20260114,2026-04-23 09:30:00,2026,1001,9002,7003,13574,16324,오러클린,화이트,...,0.40,0.11089,-0.19,0.036,-0.417,-0.435,0.2000,0,0,0.456136
4,20260115,2026-04-23 09:30:00,2026,3001,6002,8001,16434,16299,로드리게스,잭로그,...,1.45,1.61294,0.36,-0.535,3.241,0.535,-2.5385,0,0,0.421094


### Add Recent Flow Features for Reduced Core

In [8]:
# 오늘날짜에서 14일전부터 수집
pred_df_core = ut.add_recent_flow_features_for_prediction(pred_df, sleep_sec=0.15)

[games] 2026-05-19: 5 rows
[games] 2026-05-20: 5 rows
[games] 2026-05-21: 5 rows
[games] 2026-05-22: 5 rows
[games] 2026-05-23: 5 rows
[games] 2026-05-24: 5 rows
[games] 2026-05-25: 0 rows
[games] 2026-05-26: 5 rows
[games] 2026-05-27: 5 rows
[games] 2026-05-28: 5 rows
[games] 2026-05-29: 5 rows
[games] 2026-05-30: 5 rows
[games] 2026-05-31: 5 rows
[games] 2026-06-01: 0 rows
[recent] lookback_start = 2026-05-19 | completed_games_by_team = {'10001': 12, '1001': 10, '11001': 10, '12001': 11, '2002': 11, '3001': 10, '5002': 10, '6002': 11, '7002': 10, '9002': 11}
missing feature cols after recent merge: ['lineup_OPS_diff', 'lineup_OBP_diff']


c:\Users\User\Desktop\statiz_prediction\utils.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(all_dfs, ignore_index=True).drop_duplicates()


,game_id,game_date,home_team_code,away_team_code,recent_10_win_rate_diff,recent_5_run_diff_avg_diff,recent_5_runs_against_avg_diff,rest_days_diff,prev_game_win_diff
0,20260281,2026-06-02 09:30:00,6002,7002,-0.166667,-4.4,0.6,1.0,0.0
1,20260282,2026-06-02 09:30:00,9002,10001,-0.300000,1.0,0.0,0.0,0.0
2,20260283,2026-06-02 09:30:00,12001,5002,-0.066667,0.6,-1.0,0.0,0.0
3,20260284,2026-06-02 09:30:00,1001,11001,0.333333,5.0,-3.2,1.0,-1.0
4,20260285,2026-06-02 09:30:00,2002,3001,0.144444,2.4,-2.4,0.0,0.0


### Add elo features
- daily update needed

In [9]:
sys.path.append(r"C:\Users\User\Desktop\statiz_prediction")
games_with_elo, updated_snapshot = elo.update_team_elo_snapshot_2026() # print 생략됨

pred_df_elo = elo.merge_elo(pred_df_core)

UPDATE_TARGET_DATE: 2026-06-02
[games] 2026-05-31: 5 rows
[games] 2026-06-01: 0 rows


,game_id,game_date,home_team_code,away_team_code,elo_result_home,home_elo_pre,away_elo_pre,elo_diff,home_elo_post,away_elo_post
0,20260276,2026-05-31 05:00:00,5002,2002,1.0,1562.932279,1531.374702,31.557578,1571.318650,1522.988332
1,20260277,2026-05-31 05:00:00,10001,12001,0.0,1429.329417,1540.135941,-110.806524,1421.750110,1547.715248
2,20260278,2026-05-31 05:00:00,7002,9002,1.0,1535.181875,1432.005164,103.176711,1541.651389,1425.535651
3,20260279,2026-05-31 05:00:00,1001,6002,1.0,1552.031504,1501.671744,50.359760,1559.895863,1493.807385
4,20260280,2026-05-31 05:00:00,11001,3001,1.0,1445.521781,1469.815593,-24.293812,1455.501455,1459.835919


,team_code,elo,latest_update_date
0,10001,1421.750110,2026-05-31
1,1001,1559.895863,2026-05-31
2,11001,1455.501455,2026-05-31
3,12001,1547.715248,2026-05-31
4,2002,1522.988332,2026-05-31
5,3001,1459.835919,2026-05-31
6,5002,1571.318650,2026-05-31
7,6002,1493.807385,2026-05-31
8,7002,1541.651389,2026-05-31
9,9002,1425.535651,2026-05-31


saved: C:\Users\User\Desktop\statiz_prediction\team_elo_snapshot_2026.csv
updated latest_update_date: 2026-05-31


,game_id,home_team_code,away_team_code,home_elo_pre,away_elo_pre,elo_diff,elo_exp_home
0,20260281,6002,7002,1493.807385,1541.651389,-47.844003,0.467172
1,20260282,9002,10001,1425.535651,1421.750110,3.785541,0.541331
2,20260283,12001,5002,1547.715248,1571.318650,-23.603402,0.502010
3,20260284,1001,11001,1559.895863,1455.501455,104.394408,0.678056
4,20260285,2002,3001,1522.988332,1459.835919,63.152413,0.624208


missing elo rows: 0


In [9]:
pred_df_elo

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,recent_10_run_diff_avg_diff,rest_days_diff,prev_game_win_diff,home_elo_pre,latest_update_date_x,away_elo_pre,latest_update_date_y,elo_diff,elo_exp_home,home_win_proba
0,20260261,2026-05-28 09:30:00,2026,6002,12001,1001,13061,11308,곽빈,고영표,...,0.700000,0.0,-1.0,1485.214596,2026-05-27,1513.726272,2026-05-27,-28.511676,0.494946,0.577884
1,20260262,2026-05-28 09:30:00,2026,10001,2002,1003,16339,15063,로젠버그,황동하,...,-2.500000,0.0,-1.0,1454.063089,2026-05-27,1541.467278,2026-05-27,-87.404190,0.411147,0.702044
2,20260263,2026-05-28 09:30:00,2026,9002,1001,2001,16637,11379,긴지로,최원태,...,-2.900000,0.0,-1.0,1453.414287,2026-05-27,1570.990829,2026-05-27,-117.576542,0.369838,0.293886
3,20260264,2026-05-28 09:30:00,2026,3001,5002,8001,14792,14134,김진욱,이정용,...,1.000000,0.0,-1.0,1458.337861,2026-05-27,1556.697400,2026-05-27,-98.359538,0.395969,0.584173
4,20260265,2026-05-28 09:30:00,2026,11001,7002,8005,14593,16424,김태경,왕옌청,...,-2.555556,0.0,0.0,1453.967540,2026-05-27,1512.120848,2026-05-27,-58.153308,0.452433,0.489174


### logit_elo_full

In [15]:
# feature 순서 맞추기
missing_elo_cols = [c for c in feature_cols_elo if c not in pred_df_elo.columns]
if missing_elo_cols:
    raise KeyError(f"elo full 모델 입력 컬럼이 pred_df_elo에 없습니다: {missing_elo_cols}")

X_pred = pred_df_elo[feature_cols_elo].copy()
X_pred = X_pred.replace([np.inf, -np.inf], np.nan).fillna(0)

# 홈팀 승리 확률 예측
pred_df_elo["home_win_proba"] = loaded_model_elo.predict_proba(X_pred)[:, 1]

# 보기 좋게 정리
result_df = pred_df_elo[
    [
        "game_id",
        "game_date",
        "home_team_code",
        "away_team_code",
        "home_win_proba"
    ]
].copy()

display(result_df)

,game_id,game_date,home_team_code,away_team_code,home_win_proba
0,20260256,2026-05-27 09:30:00,6002,12001,0.422398
1,20260257,2026-05-27 09:30:00,10001,2002,0.380965
2,20260258,2026-05-27 09:30:00,9002,1001,0.207363
3,20260259,2026-05-27 09:30:00,3001,5002,0.599182
4,20260260,2026-05-27 09:30:00,11001,7002,0.444059


In [16]:
## 제출 형태

submit_df = result_df[["game_id", "home_win_proba"]].copy()

# 소수점 4자리 반올림 후 *100
submit_df["home_win_proba"] = submit_df["home_win_proba"].round(4) * 100

print(submit_df)

    game_id  home_win_proba
0  20260256           42.24
1  20260257           38.10
2  20260258           20.74
3  20260259           59.92
4  20260260           44.41


### logit_elo_sp

In [10]:
# feature 순서 맞추기
missing_elo_cols = [c for c in feature_cols_elo_sp if c not in pred_df_elo.columns]
if missing_elo_cols:
    raise KeyError(f"elo full 모델 입력 컬럼이 pred_df_elo에 없습니다: {missing_elo_cols}")

X_pred = pred_df_elo[feature_cols_elo_sp].copy()
X_pred = X_pred.replace([np.inf, -np.inf], np.nan).fillna(0)

# 홈팀 승리 확률 예측
pred_df_elo["home_win_proba"] = loaded_model_elo_sp.predict_proba(X_pred)[:, 1]

# 보기 좋게 정리
result_df = pred_df_elo[
    [
        "game_id",
        "game_date",
        "home_team_code",
        "away_team_code",
        "home_win_proba"
    ]
].copy()

display(result_df)

,game_id,game_date,home_team_code,away_team_code,home_win_proba
0,20260281,2026-06-02 09:30:00,6002,7002,0.652079
1,20260282,2026-06-02 09:30:00,9002,10001,0.382966
2,20260283,2026-06-02 09:30:00,12001,5002,0.414685
3,20260284,2026-06-02 09:30:00,1001,11001,0.735965
4,20260285,2026-06-02 09:30:00,2002,3001,0.553892


In [11]:
## 제출 형태

submit_df = result_df[["game_id", "home_win_proba"]].copy()

# 소수점 4자리 반올림 후 *100
submit_df["home_win_proba"] = submit_df["home_win_proba"].round(4) * 100

print(submit_df)

    game_id  home_win_proba
0  20260281           65.21
1  20260282           38.30
2  20260283           41.47
3  20260284           73.60
4  20260285           55.39


### Logit_core_full

In [9]:
# feature 순서 맞추기
missing_core_cols = [c for c in feature_cols_core if c not in pred_df_core.columns]
if missing_core_cols:
    raise KeyError(f"reduced core 모델 입력 컬럼이 pred_df_core에 없습니다: {missing_core_cols}")

X_pred = pred_df_core[feature_cols_core].copy()
X_pred = X_pred.replace([np.inf, -np.inf], np.nan).fillna(0)

# 홈팀 승리 확률 예측
pred_df_core["home_win_proba"] = loaded_model_core.predict_proba(X_pred)[:, 1]

# 보기 좋게 정리
result_df = pred_df_core[
    [
        "game_id",
        "game_date",
        "home_team_code",
        "away_team_code",
        "home_win_proba"
    ]
].copy()

display(result_df)

,game_id,game_date,home_team_code,away_team_code,home_win_proba
0,20260213,2026-05-16 05:00:00,12001,7002,0.532549
1,20260211,2026-05-16 08:00:00,6002,3001,0.607758
2,20260212,2026-05-16 08:00:00,9002,5002,0.572433
3,20260214,2026-05-16 08:00:00,1001,2002,0.686381
4,20260215,2026-05-16 08:00:00,11001,10001,0.392706


In [16]:
## 제출 형태

submit_df = result_df[["game_id", "home_win_proba"]].copy()

# 소수점 4자리 반올림 후 *100
submit_df["home_win_proba"] = submit_df["home_win_proba"].round(4) * 100

print(submit_df)

    game_id  home_win_proba
0  20260226           63.31
1  20260227           56.84
2  20260228           50.47
3  20260229           42.89
4  20260230           43.01


### Logit_core_sp

In [12]:
# feature 순서 맞추기
missing_core_cols_sp = [c for c in feature_cols_core_sp if c not in pred_df_core.columns]
if missing_core_cols_sp:
    raise KeyError(f"reduce core SP 모델 입력 컬럼이 pred_df_core에 없습니다: {missing_core_cols_sp}")

X_pred = pred_df_core[feature_cols_core_sp].copy()
X_pred = X_pred.replace([np.inf, -np.inf], np.nan).fillna(0)

# 홈팀 승리 확률 예측
pred_df_core["home_win_proba"] = loaded_model_core_sp.predict_proba(X_pred)[:, 1]

# 보기 좋게 정리
result_df = pred_df_core[
    [
        "game_id",
        "game_date",
        "home_team_code",
        "away_team_code",
        "home_win_proba"
    ]
].copy()

display(result_df)

,game_id,game_date,home_team_code,away_team_code,home_win_proba
0,20260274,2026-05-30 05:00:00,1001,6002,0.617200
1,20260271,2026-05-30 08:00:00,5002,2002,0.332748
2,20260272,2026-05-30 08:00:00,10001,12001,1.000000
3,20260273,2026-05-30 08:00:00,7002,9002,0.635162
4,20260275,2026-05-30 08:00:00,11001,3001,0.770321


In [18]:
## 제출 형태

submit_df = result_df[["game_id", "home_win_proba"]].copy()

# 소수점 4자리 반올림 후 *100
submit_df["home_win_proba"] = submit_df["home_win_proba"].round(4) * 100

print(submit_df)

    game_id  home_win_proba
0  20260226           61.51
1  20260227           58.24
2  20260228           48.56
3  20260229           44.09
4  20260230           45.68


### Logit_full

In [13]:
# feature 순서 맞추기
X_pred = pred_df[feature_cols].copy()

# 홈팀 승리 확률 예측
pred_df["home_win_proba"] = loaded_model.predict_proba(X_pred)[:, 1]

# 보기 좋게 정리
result_df = pred_df[
    [
        "game_id",
        "game_date",
        "home_team_code",
        "away_team_code",
        "home_win_proba"
    ]
].copy()

display(result_df)

,game_id,game_date,home_team_code,away_team_code,home_win_proba
0,20260126,2026-04-26 05:00:00,6002,5002,0.480104
1,20260127,2026-04-26 05:00:00,10001,1001,0.408356
2,20260128,2026-04-26 05:00:00,9002,12001,0.290651
3,20260129,2026-04-26 05:00:00,7002,11001,0.399529
4,20260130,2026-04-26 05:00:00,2002,3001,0.265862


### logit_model_sp

In [14]:
# feature 순서 맞추기
X_pred_sp = pred_df[feature_cols_sp].copy()

# 홈팀 승리 확률 예측
pred_df["home_win_proba"] = loaded_model_sp.predict_proba(X_pred_sp)[:, 1]

# 보기 좋게 정리
result_df = pred_df[
    [
        "game_id",
        "game_date",
        "home_team_code",
        "away_team_code",
        "home_win_proba"
    ]
].copy()

display(result_df)

,game_id,game_date,home_team_code,away_team_code,home_win_proba
0,20260274,2026-05-30 05:00:00,1001,6002,0.578818
1,20260271,2026-05-30 08:00:00,5002,2002,0.363217
2,20260272,2026-05-30 08:00:00,10001,12001,1.000000
3,20260273,2026-05-30 08:00:00,7002,9002,0.592638
4,20260275,2026-05-30 08:00:00,11001,3001,0.782047


In [ ]:
## 제출 형태

submit_df = result_df[["game_id", "home_win_proba"]].copy()

# 소수점 4자리 반올림 후 *100
submit_df["home_win_proba"] = submit_df["home_win_proba"].round(4) * 100

print(submit_df)

    game_id  home_win_proba
0  20260111           25.46
1  20260112           27.48
2  20260113           65.15
3  20260114           45.61
4  20260115           42.11


### API Submit

In [12]:
# 예측 결과 로그 저장 at local

sv.append_prediction_log(
    pred_df=result_df,
    file_path="prediction_log.xlsx",
    sheet_name="Raw_Submission_Log",
    dedup=True,
    model_name="elo_sp", # 모델명 기록 (elo_full, elo_sp, core_full, core_sp, full, sp, non)
)

Saved log: prediction_log.xlsx
Total rows: 240


c:\Users\User\Desktop\statiz_prediction\save_log.py:195: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat([existing_without_duplicates, new_rows], ignore_index=True)


,game_id,game_date,home_team,away_team,home_team_code,away_team_code,home_win_percent,home_win_proba,model_name,result,correct
0,20260046,2026-04-08,두산,키움,6002,10001,88.72,0.887200,full,1,1.0
1,20260047,2026-04-08,SSG,한화,9002,7002,90.58,0.905800,full,0,0.0
2,20260048,2026-04-08,KIA,삼성,2002,1001,34.54,0.345400,full,1,0.0
3,20260049,2026-04-08,롯데,KT,3001,12001,8.49,0.084900,full,1,0.0
4,20260050,2026-04-08,NC,LG,11001,5002,12.38,0.123800,full,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
235,20260281,2026-06-02,두산,한화,6002,7002,65.21,0.652079,elo_sp,NaN,NaN
236,20260282,2026-06-02,SSG,키움,9002,10001,38.30,0.382966,elo_sp,NaN,NaN
237,20260283,2026-06-02,KT,LG,12001,5002,41.47,0.414685,elo_sp,NaN,NaN
238,20260284,2026-06-02,삼성,NC,1001,11001,73.60,0.735965,elo_sp,NaN,NaN


In [13]:
submit_result_df = sb.submit_predictions_df(
    submit_df,
    path="prediction/savePrediction",     
    sleep_sec=0.2,
    verbose=True
)

display(submit_result_df)

[SUBMIT OK] s_no=20260281, percent=65.21, result_cd=100, result_msg=저장 성공
[SUBMIT OK] s_no=20260282, percent=38.30, result_cd=100, result_msg=저장 성공
[SUBMIT OK] s_no=20260283, percent=41.47, result_cd=100, result_msg=저장 성공
[SUBMIT OK] s_no=20260284, percent=73.60, result_cd=100, result_msg=저장 성공
[SUBMIT OK] s_no=20260285, percent=55.39, result_cd=100, result_msg=저장 성공


,game_id,percent,result_cd,result_msg,raw_response
0,20260281,65.21,100,저장 성공,"{'percent': 65.21, 's_no': '20260281', 'code':..."
1,20260282,38.30,100,저장 성공,"{'percent': 38.3, 's_no': '20260282', 'code': ..."
2,20260283,41.47,100,저장 성공,"{'percent': 41.47, 's_no': '20260283', 'code':..."
3,20260284,73.60,100,저장 성공,"{'percent': 73.6, 's_no': '20260284', 'code': ..."
4,20260285,55.39,100,저장 성공,"{'percent': 55.39, 's_no': '20260285', 'code':..."
